# Data Description
- Background: Unit 1.7 補充資料驗證：行人穿越線圖資。提供機關：臺北市交通局交通管制工程處；官方來源：https://data.taipei/dataset/detail?id=4d6bd0c6-9f56-408b-bb96-ac41134a95d5。原始來源對應與官方最新資源bytes完全相同為不同命題；除歷史界內包已逐byte雜湊對應外，未重新下載認證其他最新資源。
- date received: 學生於本單元提供；實際下載／收到日期未提供。首次本機檢視2026-11-26，不將今日當下載日。
- Path to data file: 官方來源及版本如上；精確本機容器如下，解壓至data/temp僅為衍生暫存。
  - `data/raw/(交工處)行人穿越線圖資_202407121519.zip`；2,005,308 bytes；SHA-256 `a7f4241ea1ecccfb09050fb38f4d8537a7f5ec1d4134eb7b8929217601ba0493`。
- Unit of observation: 六個明示行穿線面層之標線構件，不預設完整設施。
- Sample period: 2024-07-12封裝標記，非全年有效或2025新增完整紀錄；RDDATE需排除1899占位；發布／更新：目前資源頁更新2026-03-19；本機ZIP成員時間2024-07-12。ZIP時間、檔名時間、觀測／設置時間分開，不由檔名猜全年存量。
- Known issues: 六個明示行穿線圖層，Polygon構件不先假定完整設施；長寬欄0與單位未證，面積以有效幾何聯集診斷。缺失、重複、坐標及CRS由下方實讀輸出；沒有完整性證據的0僅為本檔未觀測。授權：政府資料開放授權條款第1版；依官方頁，來源與版本須保留。
- Definition for each variable: 下方逐欄型態、缺失、觀測域與定義表。已知官方欄意與本次診斷欄分開；未獲欄位碼表支持者明記未知，不依名稱猜設施單位。
- Data path: 上列每個獨立原始容器；實際讀取成員列於Summary Stats。
- Role: raw。
- Verification status: 以本Notebook乾淨kernel實際輸出與執行metadata為準；不認證已清理。
- SHA-256: 上列每檔固定綁定，載入前逐項重核。
- Next processing notebook: 本單元僅診斷，Chapter 2尚未建立；[驗證一](../code/taipei_supplementary_accident_boundary_validation.ipynb)、[驗證二](../code/taipei_transportation_infrastructure_validation.ipynb)。

Author: 詹明翰、謝子詮、曾翊涵；Created At / Last Modified At: 2026-11-26。

Primary Owner：謝子詮（DE）；Collaborator / Analytical Reviewer：詹明翰（DA）；Project Reviewer：曾翊涵（PM）。共同作者與職務不代表已完成個別工作。本輪由Codex協助執行，成員驗收與審查尚待完成。


In [1]:
from pathlib import Path
import sys, hashlib, json, zipfile
import pandas as pd
from IPython.display import display, Markdown
for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate/'AGENTS.md').is_file() and (candidate/'templates').is_dir():
        PROJECT_ROOT=candidate;break
else:raise FileNotFoundError('請從本研究工作包內執行')
def fingerprint(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for b in iter(lambda:f.read(4*1024*1024),b''):h.update(b)
    return h.hexdigest()
assert fingerprint(PROJECT_ROOT/'data/code/unit17_validation_config.json')=='7554b54042b43dd8fd2ec2c5cfa6f218e9c6acae2376ff7e0aee0e14e73a17a7'
assert fingerprint(PROJECT_ROOT/'data/code/unit17_validation_support.py')=='1a18dd248daf08074a5ba7ec659a489fe7b17852195259e8084727d4fd6f3080'
sys.path.insert(0,str(PROJECT_ROOT/'data/code'))
from unit17_validation_support import Context, show, safe_text
pd.set_option('display.max_columns',None)
pd.set_option('display.max_colwidth',160)
ctx=Context(PROJECT_ROOT,'7554b54042b43dd8fd2ec2c5cfa6f218e9c6acae2376ff7e0aee0e14e73a17a7')
print('Python:',sys.version,'Config:','7554b54042b43dd8fd2ec2c5cfa6f218e9c6acae2376ff7e0aee0e14e73a17a7','Helper:','1a18dd248daf08074a5ba7ec659a489fe7b17852195259e8084727d4fd6f3080')


Python: 3.12.2 (tags/v3.12.2:6abddd9, Feb  6 2024, 21:26:36) [MSC v.1937 64 bit (AMD64)] Config: 7554b54042b43dd8fd2ec2c5cfa6f218e9c6acae2376ff7e0aee0e14e73a17a7 Helper: 1a18dd248daf08074a5ba7ec659a489fe7b17852195259e8084727d4fd6f3080


## Change for Your Own Data

使用已核對精確路徑及固定SHA-256；不選第一個glob或最新檔。CP950非法bytes只作可逆診斷；大檔分批讀取、OSM僅研究區2km bbox，保留全國母體數。

In [2]:
FAMILY='crossings'
INPUTS=ctx.family_items(FAMILY)
for item in INPUTS:ctx.file(item['path'])
frames=ctx.load_family(FAMILY)

## Summary

每個容器、實際成員與載入範圍分列；沒有將暫存解壓檔當原始下載資料。

In [3]:
show('固定輸入身分',pd.DataFrame([{k:v for k,v in r.items() if k not in ['members','encoding_checks']} for r in INPUTS]))
show('讀取範圍與CRS／編碼',pd.DataFrame(ctx.summaries[FAMILY]))

**固定輸入身分**

,family,path,bytes,sha256,status
0,crossings,data/raw/(交工處)行人穿越線圖資_202407121519.zip,2005308,a7f4241ea1ecccfb09050fb38f4d8537a7f5ec1d4134eb7b8929217601ba0493,present


**讀取範圍與CRS／編碼**

,path,member,total_features,loaded_rows,crs,types,bounds,invalid,null_geometry,empty
0,data/raw/(交工處)行人穿越線圖資_202407121519.zip,斑馬紋行人穿越線/grapline_09.shp,364,364,EPSG:3826,"{'Polygon': 344, 'MultiPolygon': 20}","[297023.5054, 2763293.77612, 312913.02, 2783186.96592]",0,0,0
1,data/raw/(交工處)行人穿越線圖資_202407121519.zip,斑馬紋行人穿越線/grapline_A9.shp,393,393,EPSG:3826,"{'Polygon': 381, 'MultiPolygon': 12}","[300301.1823, 2764819.413737, 312901.1796, 2782410.29102]",0,0,0
2,data/raw/(交工處)行人穿越線圖資_202407121519.zip,枕木紋行人穿越線-間隔80cm/grapline_18.shp,1659,1659,EPSG:3826,{'Polygon': 1659},"[297093.662574, 2762403.742016, 312984.201766, 2784527.353119]",0,0,0
3,data/raw/(交工處)行人穿越線圖資_202407121519.zip,枕木紋行人穿越線-間隔80cm-/grapline_B8.shp,1451,1451,EPSG:3826,"{'Polygon': 1448, 'MultiPolygon': 3}","[297036.8754, 2763269.210397, 312604.4025, 2781978.83293]",0,0,0
4,data/raw/(交工處)行人穿越線圖資_202407121519.zip,枕木紋行人穿越線-間隔40cm/grapline_08.shp,7354,7354,EPSG:3826,"{'Polygon': 7337, 'MultiPolygon': 17}","[296696.3241, 2762392.64312, 313003.7077, 2785558.021419]",0,0,0
5,data/raw/(交工處)行人穿越線圖資_202407121519.zip,枕木紋行人穿越線-間隔40cm-/grapline_A8.shp,8422,8422,EPSG:3826,"{'Polygon': 8402, 'MultiPolygon': 20}","[296793.1661, 2762389.96892, 312858.6852, 2785784.333819]",0,0,0


### Sample

各實際讀取表前十列；全國事故為臺北篩選後第一當事人，不代表全國前十列。OSM為bbox空間子集。

In [4]:
for source,frame in frames.items():
    show(source,frame.drop(columns='geometry',errors='ignore').head(10),block=10)

**data/raw/(交工處)行人穿越線圖資_202407121519.zip::斑馬紋行人穿越線/grapline_09.shp**

,keyid,dxf_angle,rditcd,rdcode,licode,liarea,rddate,rdsncn,sirean,rdctnb,editby,rdlblg,rdlbwt,workc_keyi,streetview,streetvi_1,streetvi_2,streetvi_3,streetvi_4,workc_in_u,workc_in_t,edittime,zlocation,parefe,walkerroad,proj_name,proj_no,rpfrom,rparea,rplocation,rpdate,walkerro_1,walkerstat,roadid,_source_row,_source_path,_source_member
fid,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
0,3143,0.0,0.0,None,09,0.0,2008-11-27,NaN,NaN,None,董映琳,0.0,0.0,0,1.352870e+07,2.888299e+06,-31.042314,-0.055751,2.98,0,NaT,2023-01-16,01,FIX,None,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,1,data/raw/(交工處)行人穿越線圖資_202407121519.zip,斑馬紋行人穿越線/grapline_09.shp
1,1797,0.0,0.0,None,09,0.0,2005-07-12,0101,01,None,董映琳,0.0,0.0,0,1.352642e+07,2.881459e+06,-31.042314,-0.055751,2.98,0,NaT,2023-03-09,01,FIX,None,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,2,data/raw/(交工處)行人穿越線圖資_202407121519.zip,斑馬紋行人穿越線/grapline_09.shp
2,1882,0.0,0.0,None,09,0.0,2005-07-12,0101,01,None,董映琳,0.0,0.0,0,1.352686e+07,2.884929e+06,-31.042314,-0.055751,2.98,0,NaT,2023-03-13,01,FIX,None,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,3,data/raw/(交工處)行人穿越線圖資_202407121519.zip,斑馬紋行人穿越線/grapline_09.shp
3,855,0.0,0.0,None,09,0.0,1899-12-30,0101,01,None,NaN,0.0,0.0,0,1.352480e+07,2.877769e+06,-31.042314,-0.055751,2.98,0,NaT,2016-10-07,01,FIX,None,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,4,data/raw/(交工處)行人穿越線圖資_202407121519.zip,斑馬紋行人穿越線/grapline_09.shp
4,925,0.0,0.0,None,09,0.0,1899-12-30,0101,01,None,NaN,0.0,0.0,0,1.352486e+07,2.877778e+06,-31.042314,-0.055751,2.98,0,NaT,2016-10-07,01,FIX,None,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,5,data/raw/(交工處)行人穿越線圖資_202407121519.zip,斑馬紋行人穿越線/grapline_09.shp
5,935,0.0,0.0,None,09,0.0,1899-12-30,0101,01,None,NaN,0.0,0.0,0,1.352492e+07,2.879440e+06,-31.042314,-0.055751,2.98,0,NaT,2016-10-07,01,FIX,None,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,6,data/raw/(交工處)行人穿越線圖資_202407121519.zip,斑馬紋行人穿越線/grapline_09.shp
6,1021,0.0,0.0,None,09,0.0,1899-12-30,0101,01,None,NaN,0.0,0.0,0,1.352527e+07,2.891757e+06,-31.042314,-0.055751,2.98,0,NaT,2016-10-07,01,FIX,None,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,7,data/raw/(交工處)行人穿越線圖資_202407121519.zip,斑馬紋行人穿越線/grapline_09.shp
7,1315,0.0,0.0,None,09,0.0,2005-07-12,NaN,NaN,None,NaN,0.0,0.0,0,1.352559e+07,2.892107e+06,-31.042314,-0.055751,2.98,0,NaT,2016-10-07,01,FIX,None,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,8,data/raw/(交工處)行人穿越線圖資_202407121519.zip,斑馬紋行人穿越線/grapline_09.shp
8,1873,0.0,0.0,None,09,0.0,2005-07-12,0101,01,None,NaN,0.0,0.0,0,1.352736e+07,2.880739e+06,-31.042314,-0.055751,2.98,0,NaT,2016-10-07,01,FIX,None,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,9,data/raw/(交工處)行人穿越線圖資_202407121519.zip,斑馬紋行人穿越線/grapline_09.shp


**data/raw/(交工處)行人穿越線圖資_202407121519.zip::斑馬紋行人穿越線/grapline_A9.shp**

,keyid,dxf_angle,rditcd,rdcode,licode,liarea,rddate,rdsncn,sirean,rdctnb,editby,rdlblg,rdlbwt,workc_keyi,streetview,streetvi_1,streetvi_2,streetvi_3,streetvi_4,workc_in_u,workc_in_t,edittime,zlocation,parefe,walkerroad,proj_name,proj_no,rpfrom,rparea,rplocation,rpdate,walkerro_1,walkerstat,roadid,_source_row,_source_path,_source_member
fid,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
0,3171,0.0,0.0,None,A9,0.0,2014-10-03,0101,01,None,董映琳,0.0,0.0,0,1.353110e+07,2.888178e+06,-31.042314,-0.055751,2.98,0,NaT,2023-02-10,01,FIX,None,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,1,data/raw/(交工處)行人穿越線圖資_202407121519.zip,斑馬紋行人穿越線/grapline_A9.shp
1,2589,0.0,0.0,None,A9,0.0,2005-07-12,0101,01,None,董映琳,0.0,0.0,0,1.352909e+07,2.887734e+06,-31.042314,-0.055751,2.98,0,NaT,2023-02-16,01,FIX,None,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,2,data/raw/(交工處)行人穿越線圖資_202407121519.zip,斑馬紋行人穿越線/grapline_A9.shp
2,3489,0.0,0.0,None,A9,0.0,1999-09-16,0101,01,None,NaN,0.0,0.0,0,1.353294e+07,2.878981e+06,-31.042314,-0.055751,2.98,0,NaT,2016-10-07,01,FIX,None,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,3,data/raw/(交工處)行人穿越線圖資_202407121519.zip,斑馬紋行人穿越線/grapline_A9.shp
3,3496,0.0,0.0,None,A9,0.0,1999-09-16,0101,01,None,NaN,0.0,0.0,0,1.353849e+07,2.884158e+06,-31.042314,-0.055751,2.98,0,NaT,2016-10-07,01,FIX,None,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,4,data/raw/(交工處)行人穿越線圖資_202407121519.zip,斑馬紋行人穿越線/grapline_A9.shp
4,3499,0.0,0.0,None,A9,0.0,1999-09-16,0101,01,None,NaN,0.0,0.0,0,1.353715e+07,2.884744e+06,-31.042314,-0.055751,2.98,0,NaT,2016-10-07,01,FIX,None,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,5,data/raw/(交工處)行人穿越線圖資_202407121519.zip,斑馬紋行人穿越線/grapline_A9.shp
5,3505,0.0,0.0,None,A9,0.0,2013-04-09,0101,01,None,NaN,0.0,0.0,0,1.353772e+07,2.884405e+06,-31.042314,-0.055751,2.98,0,NaT,2016-10-07,01,FIX,None,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,6,data/raw/(交工處)行人穿越線圖資_202407121519.zip,斑馬紋行人穿越線/grapline_A9.shp
6,4174,0.0,0.0,None,A9,0.0,2007-02-01,0101,01,None,NaN,0.0,0.0,0,1.352519e+07,2.879348e+06,-31.042314,-0.055751,2.98,0,NaT,2016-10-07,01,FIX,None,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,7,data/raw/(交工處)行人穿越線圖資_202407121519.zip,斑馬紋行人穿越線/grapline_A9.shp
7,4607,0.0,0.0,None,A9,0.0,2007-01-22,0101,01,None,NaN,0.0,0.0,0,1.352534e+07,2.891577e+06,-31.042314,-0.055751,2.98,0,NaT,2016-10-07,01,FIX,None,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,8,data/raw/(交工處)行人穿越線圖資_202407121519.zip,斑馬紋行人穿越線/grapline_A9.shp
8,4610,0.0,0.0,None,A9,0.0,2007-01-22,0101,01,None,NaN,0.0,0.0,0,1.352526e+07,2.891747e+06,-31.042314,-0.055751,2.98,0,NaT,2016-10-07,01,FIX,None,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,9,data/raw/(交工處)行人穿越線圖資_202407121519.zip,斑馬紋行人穿越線/grapline_A9.shp


**data/raw/(交工處)行人穿越線圖資_202407121519.zip::枕木紋行人穿越線-間隔80cm/grapline_18.shp**

,keyid,dxf_angle,rditcd,rdcode,licode,liarea,rddate,rdsncn,sirean,rdctnb,editby,rdlblg,rdlbwt,workc_keyi,streetview,streetvi_1,streetvi_2,streetvi_3,streetvi_4,workc_in_u,workc_in_t,edittime,zlocation,parefe,walkerroad,proj_name,proj_no,rpfrom,rparea,rplocation,rpdate,walkerro_1,walkerstat,roadid,_source_row,_source_path,_source_member
fid,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
0,21534,0.0,0.0,None,18,0.0,2022-03-16,NaN,01,None,董映琳,7.0,40.0,17305.0,0.000000e+00,0.000000e+00,0.000000,0.000000,0.00,299.0,2022-04-19,2023-01-07,01,FIX,None,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,1,data/raw/(交工處)行人穿越線圖資_202407121519.zip,枕木紋行人穿越線-間隔80cm/grapline_18.shp
1,20967,0.0,0.0,None,18,0.0,2019-03-02,NaN,NaN,None,董映琳,0.0,0.0,0.0,1.352854e+07,2.888699e+06,-31.042314,-0.055751,2.98,0.0,NaT,2023-01-16,01,FIX,None,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,2,data/raw/(交工處)行人穿越線圖資_202407121519.zip,枕木紋行人穿越線-間隔80cm/grapline_18.shp
2,20790,0.0,0.0,None,18,0.0,2003-10-23,NaN,01,None,董映琳,11.0,40.0,5678.0,0.000000e+00,0.000000e+00,0.000000,0.000000,0.00,297.0,2018-10-12,2023-01-18,01,FIX,None,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,3,data/raw/(交工處)行人穿越線圖資_202407121519.zip,枕木紋行人穿越線-間隔80cm/grapline_18.shp
3,20537,0.0,0.0,None,18,0.0,2003-07-24,NaN,01,None,董映琳,3.0,10.0,2905.0,0.000000e+00,0.000000e+00,0.000000,0.000000,0.00,297.0,2018-04-10,2023-01-17,01,FIX,None,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,4,data/raw/(交工處)行人穿越線圖資_202407121519.zip,枕木紋行人穿越線-間隔80cm/grapline_18.shp
4,21390,0.0,0.0,None,18,0.0,2020-03-16,NaN,01,None,董映琳,0.0,40.0,12427.0,0.000000e+00,0.000000e+00,0.000000,0.000000,0.00,158.0,2021-05-11,2023-01-07,01,FIX,None,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,5,data/raw/(交工處)行人穿越線圖資_202407121519.zip,枕木紋行人穿越線-間隔80cm/grapline_18.shp
5,38406,NaN,NaN,None,18,NaN,NaT,NaN,01,None,沈敬莘,15.0,40.0,24197.0,NaN,NaN,NaN,NaN,NaN,299.0,2023-01-13,2023-01-13,01,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,6,data/raw/(交工處)行人穿越線圖資_202407121519.zip,枕木紋行人穿越線-間隔80cm/grapline_18.shp
6,20635,0.0,0.0,None,18,0.0,2003-07-24,NaN,01,None,董映琳,5.0,10.0,3185.0,0.000000e+00,0.000000e+00,0.000000,0.000000,0.00,158.0,2018-04-16,2023-02-01,01,FIX,None,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,7,data/raw/(交工處)行人穿越線圖資_202407121519.zip,枕木紋行人穿越線-間隔80cm/grapline_18.shp
7,38413,NaN,NaN,None,18,NaN,NaT,NaN,01,None,沈敬莘,4.0,40.0,24184.0,NaN,NaN,NaN,NaN,NaN,299.0,2023-01-13,2023-01-13,01,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,8,data/raw/(交工處)行人穿越線圖資_202407121519.zip,枕木紋行人穿越線-間隔80cm/grapline_18.shp
8,20800,0.0,0.0,None,18,0.0,2003-10-23,NaN,01,None,董映琳,13.0,40.0,5148.0,0.000000e+00,0.000000e+00,0.000000,0.000000,0.00,297.0,2018-10-12,2023-02-10,01,FIX,None,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,9,data/raw/(交工處)行人穿越線圖資_202407121519.zip,枕木紋行人穿越線-間隔80cm/grapline_18.shp


... (2 outputs left out)

**data/raw/(交工處)行人穿越線圖資_202407121519.zip::枕木紋行人穿越線-間隔40cm/grapline_08.shp**

,keyid,dxf_angle,rditcd,rdcode,licode,liarea,rddate,rdsncn,sirean,rdctnb,editby,rdlblg,rdlbwt,workc_keyi,streetview,streetvi_1,streetvi_2,streetvi_3,streetvi_4,workc_in_u,workc_in_t,edittime,zlocation,parefe,walkerroad,proj_name,proj_no,rpfrom,rparea,rplocation,rpdate,walkerro_1,walkerstat,roadid,_source_row,_source_path,_source_member
fid,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
0,8580,0.0,0.0,None,08,0.0,2013-05-16,0101,01,None,董映琳,0.0,0.0,0.0,1.352802e+07,2.890309e+06,-31.042314,-0.055751,2.98,0.0,NaT,2023-01-05,01,FIX,None,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,1,data/raw/(交工處)行人穿越線圖資_202407121519.zip,枕木紋行人穿越線-間隔40cm/grapline_08.shp
1,38288,0.0,NaN,None,08,0.0,2023-01-07,NaN,NaN,None,董映琳,0.0,0.0,NaN,1.352965e+07,2.890278e+06,-31.042314,-0.055751,2.98,NaN,NaT,2023-01-07,01,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,2,data/raw/(交工處)行人穿越線圖資_202407121519.zip,枕木紋行人穿越線-間隔40cm/grapline_08.shp
2,11276,0.0,0.0,None,08,0.0,2008-01-10,0101,01,None,董映琳,0.0,0.0,0.0,1.352944e+07,2.890252e+06,-31.042314,-0.055751,2.98,0.0,NaT,2023-01-07,01,FIX,None,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,3,data/raw/(交工處)行人穿越線圖資_202407121519.zip,枕木紋行人穿越線-間隔40cm/grapline_08.shp
3,38289,0.0,NaN,None,08,0.0,2023-01-07,NaN,NaN,None,董映琳,0.0,0.0,NaN,1.352930e+07,2.890274e+06,-31.042314,-0.055751,2.98,NaN,NaT,2023-01-07,01,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,4,data/raw/(交工處)行人穿越線圖資_202407121519.zip,枕木紋行人穿越線-間隔40cm/grapline_08.shp
4,38295,0.0,NaN,None,08,0.0,2023-01-09,NaN,NaN,None,董映琳,0.0,0.0,NaN,1.352809e+07,2.887051e+06,-31.042314,-0.055751,2.98,NaN,NaT,2023-01-09,01,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,5,data/raw/(交工處)行人穿越線圖資_202407121519.zip,枕木紋行人穿越線-間隔40cm/grapline_08.shp
5,8596,0.0,0.0,None,08,0.0,2013-05-16,0101,01,None,董映琳,0.0,0.0,0.0,1.352803e+07,2.890323e+06,-31.042314,-0.055751,2.98,0.0,NaT,2023-01-05,01,FIX,None,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,6,data/raw/(交工處)行人穿越線圖資_202407121519.zip,枕木紋行人穿越線-間隔40cm/grapline_08.shp
6,9914,0.0,0.0,None,08,0.0,2008-01-10,0101,01,None,董映琳,0.0,0.0,0.0,1.352880e+07,2.889579e+06,-31.042314,-0.055751,2.98,0.0,NaT,2023-01-07,01,FIX,None,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,7,data/raw/(交工處)行人穿越線圖資_202407121519.zip,枕木紋行人穿越線-間隔40cm/grapline_08.shp
7,9994,0.0,0.0,None,08,0.0,2013-04-11,0101,01,None,董映琳,0.0,0.0,0.0,1.352917e+07,2.890271e+06,-31.042314,-0.055751,2.98,0.0,NaT,2023-01-07,01,FIX,None,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,8,data/raw/(交工處)行人穿越線圖資_202407121519.zip,枕木紋行人穿越線-間隔40cm/grapline_08.shp
8,8162,0.0,0.0,None,08,0.0,2010-08-19,0101,01,None,董映琳,0.0,0.0,0.0,1.352825e+07,2.886588e+06,-31.042314,-0.055751,2.98,0.0,NaT,2023-01-09,01,FIX,None,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,9,data/raw/(交工處)行人穿越線圖資_202407121519.zip,枕木紋行人穿越線-間隔40cm/grapline_08.shp


**data/raw/(交工處)行人穿越線圖資_202407121519.zip::枕木紋行人穿越線-間隔40cm-/grapline_A8.shp**

,keyid,dxf_angle,rditcd,rdcode,licode,liarea,rddate,rdsncn,sirean,rdctnb,editby,rdlblg,rdlbwt,workc_keyi,streetview,streetvi_1,streetvi_2,streetvi_3,streetvi_4,workc_in_u,workc_in_t,edittime,zlocation,parefe,walkerroad,proj_name,proj_no,rpfrom,rparea,rplocation,rpdate,walkerro_1,walkerstat,roadid,_source_row,_source_path,_source_member
fid,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
0,9913,0.0,0.0,None,A8,0.0,2008-01-10,0101,01,None,董映琳,0.0,0.0,0.0,1.352881e+07,2.889571e+06,-31.042314,-0.055751,2.98,0.0,NaT,2023-01-07,01,FIX,None,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,1,data/raw/(交工處)行人穿越線圖資_202407121519.zip,枕木紋行人穿越線-間隔40cm-/grapline_A8.shp
1,21505,0.0,0.0,None,A8,0.0,2022-03-16,NaN,01,None,董映琳,10.0,40.0,18069.0,0.000000e+00,0.000000e+00,0.000000,0.000000,0.00,299.0,2022-04-19,2023-01-07,01,FIX,None,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,2,data/raw/(交工處)行人穿越線圖資_202407121519.zip,枕木紋行人穿越線-間隔40cm-/grapline_A8.shp
2,11265,0.0,0.0,None,A8,0.0,1999-06-24,NaN,02,None,董映琳,0.0,0.0,0.0,1.352949e+07,2.890053e+06,-31.042314,-0.055751,2.98,0.0,NaT,2023-01-07,01,FIX,None,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,3,data/raw/(交工處)行人穿越線圖資_202407121519.zip,枕木紋行人穿越線-間隔40cm-/grapline_A8.shp
3,38292,0.0,NaN,None,A8,0.0,2023-01-07,NaN,NaN,None,董映琳,0.0,0.0,NaN,1.352955e+07,2.889712e+06,-31.042314,-0.055751,2.98,NaN,NaT,2023-01-07,01,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,4,data/raw/(交工處)行人穿越線圖資_202407121519.zip,枕木紋行人穿越線-間隔40cm-/grapline_A8.shp
4,8212,0.0,0.0,None,A8,0.0,2010-08-19,0101,01,None,董映琳,0.0,0.0,0.0,1.352831e+07,2.886839e+06,-31.042314,-0.055751,2.98,0.0,NaT,2023-01-09,01,FIX,None,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,5,data/raw/(交工處)行人穿越線圖資_202407121519.zip,枕木紋行人穿越線-間隔40cm-/grapline_A8.shp
5,38296,0.0,NaN,None,A8,0.0,2023-01-09,NaN,NaN,None,董映琳,0.0,0.0,NaN,1.352861e+07,2.890416e+06,-31.042314,-0.055751,2.98,NaN,NaT,2023-01-09,01,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,6,data/raw/(交工處)行人穿越線圖資_202407121519.zip,枕木紋行人穿越線-間隔40cm-/grapline_A8.shp
6,38350,0.0,NaN,None,A8,0.0,2023-01-11,NaN,NaN,None,董映琳,0.0,0.0,NaN,1.352877e+07,2.889739e+06,-31.042314,-0.055751,2.98,NaN,NaT,2023-01-13,01,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,7,data/raw/(交工處)行人穿越線圖資_202407121519.zip,枕木紋行人穿越線-間隔40cm-/grapline_A8.shp
7,9977,0.0,0.0,None,A8,0.0,2013-04-11,0101,01,None,董映琳,0.0,0.0,0.0,1.352905e+07,2.890001e+06,-31.042314,-0.055751,2.98,0.0,NaT,2023-01-07,01,FIX,None,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,8,data/raw/(交工處)行人穿越線圖資_202407121519.zip,枕木紋行人穿越線-間隔40cm-/grapline_A8.shp
8,11263,0.0,0.0,None,A8,0.0,1999-06-24,NaN,02,None,董映琳,0.0,0.0,0.0,1.352951e+07,2.890053e+06,-31.042314,-0.055751,2.98,0.0,NaT,2023-01-07,01,FIX,None,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,9,data/raw/(交工處)行人穿越線圖資_202407121519.zip,枕木紋行人穿越線-間隔40cm-/grapline_A8.shp


### Summary Stats

ZIP逐成員CRC、配套、原始WKT、全部欄型態／缺失／定義、重複與數值／類別／時間診斷。保留原值及bytes。

In [5]:
_ = ctx.raw_report(FAMILY,crc=True)

,source_metadata_and_limits
title,行人穿越線圖資
provider,臺北市交通局交通管制工程處
url,https://data.taipei/dataset/detail?id=4d6bd0c6-9f56-408b-bb96-ac41134a95d5
publication,目前資源頁更新2026-03-19；本機ZIP成員時間2024-07-12
period,2024-07-12封裝標記，非全年有效或2025新增完整紀錄；RDDATE需排除1899占位
limitation,六個明示行穿線圖層，Polygon構件不先假定完整設施；長寬欄0與單位未證，面積以有效幾何聯集診斷
license,政府資料開放授權條款第1版；依官方頁，來源與版本須保留


,family,path,bytes,sha256,status
0,crossings,data/raw/(交工處)行人穿越線圖資_202407121519.zip,2005308,a7f4241ea1ecccfb09050fb38f4d8537a7f5ec1d4134eb7b8929217601ba0493,present


**ZIP原始成員清單；非解壓暫存資料來源**

,member,bytes,compressed,crc32,zip_datetime
0,斑馬紋行人穿越線/grapline_09.cpg,5,7,0e813c50,"[2024, 7, 12, 15, 19, 58]"
1,斑馬紋行人穿越線/grapline_09.dbf,450662,13864,cfad892e,"[2024, 7, 12, 15, 19, 58]"
2,斑馬紋行人穿越線/grapline_09.prj,399,256,90c78104,"[2024, 7, 12, 15, 19, 58]"
3,斑馬紋行人穿越線/grapline_09.shp,66772,15782,6d22eb0d,"[2024, 7, 12, 15, 19, 58]"
4,斑馬紋行人穿越線/grapline_09.shx,3012,1279,39672823,"[2024, 7, 12, 15, 19, 58]"
5,斑馬紋行人穿越線/grapline_A9.cpg,5,7,0e813c50,"[2024, 7, 12, 15, 19, 58]"
6,斑馬紋行人穿越線/grapline_A9.dbf,486477,15500,f3cff593,"[2024, 7, 12, 15, 19, 58]"
7,斑馬紋行人穿越線/grapline_A9.prj,399,256,90c78104,"[2024, 7, 12, 15, 19, 58]"
8,斑馬紋行人穿越線/grapline_A9.shp,71756,17027,17a1ef22,"[2024, 7, 12, 15, 19, 58]"
9,斑馬紋行人穿越線/grapline_A9.shx,3244,1351,302bae1b,"[2024, 7, 12, 15, 19, 58]"


,member,bytes,compressed,crc32,zip_datetime
18,枕木紋行人穿越線-間隔80cm-/grapline_B8.shp,198876,82037,80c29f37,"[2024, 7, 12, 15, 19, 58]"
19,枕木紋行人穿越線-間隔80cm-/grapline_B8.shx,11708,3883,59ea1036,"[2024, 7, 12, 15, 19, 58]"
20,枕木紋行人穿越線-間隔40cm/grapline_08.cpg,5,7,0e813c50,"[2024, 7, 12, 15, 19, 54]"
21,枕木紋行人穿越線-間隔40cm/grapline_08.dbf,9083312,280329,c9c29a91,"[2024, 7, 12, 15, 19, 56]"
22,枕木紋行人穿越線-間隔40cm/grapline_08.prj,399,256,90c78104,"[2024, 7, 12, 15, 19, 54]"
23,枕木紋行人穿越線-間隔40cm/grapline_08.shp,1033064,469068,8de2ff37,"[2024, 7, 12, 15, 19, 56]"
24,枕木紋行人穿越線-間隔40cm/grapline_08.shx,58932,21048,aeb98f70,"[2024, 7, 12, 15, 19, 56]"
25,枕木紋行人穿越線-間隔40cm-/grapline_A8.cpg,5,7,0e813c50,"[2024, 7, 12, 15, 19, 56]"
26,枕木紋行人穿越線-間隔40cm-/grapline_A8.dbf,10402292,318107,5d94f807,"[2024, 7, 12, 15, 19, 56]"
27,枕木紋行人穿越線-間隔40cm-/grapline_A8.prj,399,256,90c78104,"[2024, 7, 12, 15, 19, 56]"


ZIP 全成員CRC通過: data/raw/(交工處)行人穿越線圖資_202407121519.zip


... (94 outputs left out)

原始屬性完整列重複（不含診斷追溯欄） 0


**類別前12項：licode**

,licode,N,ratio_of_loaded_rows
0,A8,8422,1.0


,field,missing_or_unparseable,pre1950_placeholder,eligible_survivors_2024,eligible_survivors_2025,post2025
0,rddate,83,2,8337,8337,0


本文件只確認資料身分及可讀性；語義／日期不足者限制用途，不強行納入正式Dataset B3。人類驗收仍待完成。

In [6]:
for item in INPUTS:ctx.file(item['path'])
print('原始容器hash重核通過；正式分析資料輸出0。')

原始容器hash重核通過；正式分析資料輸出0。
